In [16]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import h5py
import json
plt.style.use('./graph_preset.mplstyle')

In [17]:
read_path = Path("./0316172637/results.h5")

In [18]:
with h5py.File(read_path, "r") as f: # read_paths[#] that you want to read
    print(f"--- Structure of {read_path} ---")

    def print_structure(name, obj):
        # データセットの場合は形状とデータ型も表示
        if isinstance(obj, h5py.Dataset):
            print(f"  {name} (Dataset) | Shape: {obj.shape}, Dtype: {obj.dtype}")
        # グループの場合はグループ名を表示
        elif isinstance(obj, h5py.Group):
            print(f"  {name} (Group)")

    f.visititems(print_structure)
    print("---------------------------------")

--- Structure of 0316172637\results.h5 ---
  input (Group)
  learning_curve (Group)
  output (Group)
  output/repeat_1 (Dataset) | Shape: (10, 9), Dtype: float32
---------------------------------


In [19]:
df_data = dict()

def store_dataset(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"  Loading: {name} | Shape: {obj.shape}")
        df = pd.DataFrame(obj[:])
        df_data[name] = df

def store_dataset(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"  Loading: {name} | Shape: {obj.shape}")

        # columns 属性があれば JSON から復元
        cols_attr = obj.attrs.get("columns", None)
        columns = None
        if cols_attr is not None:
            # 古い h5py だと bytes / np.bytes_ で返ることがある
            if isinstance(cols_attr, (bytes, np.bytes_)):
                cols_attr = cols_attr.decode("utf-8")
            columns = json.loads(cols_attr)

        # DataFrame 化（列名があれば使う）
        data = obj[:]  # dset[:, :] と同じ
        if columns is not None:
            df = pd.DataFrame(data, columns=columns)
        else:
            df = pd.DataFrame(data)

        df_data[name] = df


with h5py.File(read_path, "r") as f:
    print(f"--- Loading all datasets from {read_path} ---")
    f.visititems(store_dataset)
    print("---------------------------------------------")

print("\n--- Dictionary Keys ---")
print(list(df_data.keys()))
print("-----------------------")

--- Loading all datasets from 0316172637\results.h5 ---
  Loading: output/repeat_1 | Shape: (10, 9)
---------------------------------------------

--- Dictionary Keys ---
['output/repeat_1']
-----------------------


In [20]:
pd.set_option('display.max_rows', None)

In [21]:
df_data["output/repeat_1"]

,c,n,a,b,k,S11,Acq,gamma,best
0,-9.040000,2.420000,2.220000,3.290000,2.000000,-0.728653,NaN,NaN,-0.728653
1,-6.000000,3.000000,5.782376,2.611128,3.397696,-4.481200,NaN,NaN,-4.481200
2,-6.000000,3.000000,6.958296,7.174836,1.080209,-0.866684,NaN,NaN,-4.481200
3,-6.000000,3.000000,4.652621,5.949967,5.037632,-14.723325,NaN,NaN,-14.723325
4,-6.000000,3.000000,2.687339,4.861152,4.264513,-11.107970,NaN,NaN,-14.723325
5,-5.970085,3.005710,4.251735,5.529957,4.874759,-13.798083,15.255539,0.100000,-14.723325
6,-6.127809,2.975600,4.611423,6.646517,5.029694,-12.679592,15.445765,0.100000,-14.723325
7,-5.988319,3.002240,4.934818,5.900194,4.963480,-13.975778,14.977037,0.236577,-14.723325
8,-5.856611,3.027407,4.506991,5.996141,5.163114,-14.304487,14.992354,0.309562,-14.723325
9,-6.180147,2.851043,4.495828,5.958563,5.017653,-14.350013,15.100077,0.346196,-14.723325
